# Run experiments on PFA and check for variability in performance

## Imports and paths

In [ ]:
!pip install -q scikit-learn pandas numpy

import os
import random
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/education-ml-research"

DATASETS = {
    "ASSISTments2009": {
        "data_path": os.path.join(
            BASE_DIR,
            "ASSISTments2009",
            "skill_builder_data.csv"
        ),
        "groups_path": os.path.join(
            BASE_DIR,
            "ASSISTments2009",
            "student_irt_ability_groups.csv"
        ),
    },
    "ASSISTments2017": {
        "data_path": os.path.join(
            BASE_DIR,
            "ASSISTments2017",
            "anonymized_full_release_competition_dataset.csv"
        ),
        "groups_path": os.path.join(
            BASE_DIR,
            "ASSISTments2017",
            "student_irt_ability_groups.csv"
        ),
    },
    "Junyi": {
        "data_path": os.path.join(
            BASE_DIR,
            "Junyi",
            "Log_Problem.csv"
        ),
        "groups_path": os.path.join(
            BASE_DIR,
            "Junyi",
            "student_irt_ability_groups.csv"
        ),
    },
}

# Ensure all paths exist
for name, paths in DATASETS.items():
    print(f"\n{name}")
    for label, path in paths.items():
        status = "FOUND" if os.path.exists(path) else "MISSING"
        print(f"{label}: {status}")

NumPy: 2.1.3
Pandas: 2.2.3
Mounted at /content/drive

ASSISTments2009
data_path: FOUND
groups_path: FOUND

ASSISTments2017
data_path: FOUND
groups_path: FOUND

Junyi
data_path: FOUND
groups_path: FOUND


## Load the datasets

In [ ]:

def load_pfa_dataset(name, paths):
    # Load dataset
    if name == "ASSISTments2009":
        df = pd.read_csv(
            paths["data_path"],
            encoding="latin-1",
            low_memory=False
        )

        df = df.rename(columns={
            "user_id": "student_id",
            "skill_id": "skill_id",
            "correct": "correct",
            "order_id": "time",
        })

        df = df.dropna(
            subset=["student_id", "skill_id", "correct", "time"]
        ).copy()

        df["correct"] = pd.to_numeric(df["correct"], errors="coerce")
        df["time"] = pd.to_numeric(df["time"], errors="coerce")

    elif name == "ASSISTments2017":
        df = pd.read_csv(
            paths["data_path"],
            low_memory=False
        )

        df = df.rename(columns={
            "studentId": "student_id",
            "skill": "skill_id",
            "correct": "correct",
            "startTime": "time",
        })

        df = df.dropna(
            subset=["student_id", "skill_id", "correct", "time"]
        ).copy()

        df["skill_id"] = df["skill_id"].astype(str)

        # Exclude interactions without a usable skill
        df = df[df["skill_id"].str.lower() != "noskill"].copy()

        df["correct"] = pd.to_numeric(df["correct"], errors="coerce")

        df["time"] = pd.to_numeric(df["time"], errors="coerce")

    elif name == "Junyi":
        df = pd.read_csv(paths["data_path"])

        df = df.rename(columns={
            "uuid": "student_id",
            "ucid": "skill_id",
            "is_correct": "correct",
            "timestamp_TW": "time",
        })

        df = df.dropna(
            subset=["student_id", "skill_id", "correct", "time"]
        ).copy()

        df["correct"] = df["correct"].map({
            True: 1,
            False: 0,
            1: 1,
            0: 0
        })

        # Junyi timestamps are actual datetime values
        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        )

    else:
        raise ValueError(f"Unknown dataset: {name}")

    # Clean and standardize columns
    df = df[
        ["student_id", "skill_id", "correct", "time"]
    ].copy()

    df["correct"] = pd.to_numeric(df["correct"], errors="coerce")

    df = df.dropna(
        subset=["student_id", "skill_id", "correct", "time"]
    ).copy()

    # Only retain binary correctness labels
    df = df[df["correct"].isin([0, 1])].copy()

    # Standardize skill identifiers
    df["skill_id"] = df["skill_id"].astype(str)

    # Sort chronologically within each student
    df = df.sort_values(
        ["student_id", "time"],
        kind="stable"
    ).reset_index(drop=True)

    df["correct"] = df["correct"].astype(int)

    return df


## Check the output

In [ ]:

pfa_data = {}

for name, paths in DATASETS.items():
    standardized = load_pfa_dataset(name, paths)
    pfa_data[name] = standardized

    print(f"\n{name}")
    print("Rows:", len(standardized))
    print("Students:", standardized["student_id"].nunique())
    print("Skills:", standardized["skill_id"].nunique())
    print("Time dtype:", standardized["time"].dtype)
    print("Time range:", standardized["time"].min(), "to", standardized["time"].max())
    print("Missing values:")
    print(standardized.isna().sum())

    display(standardized.head(3))



ASSISTments2009
Rows: 459208
Students: 4163
Skills: 123
Time dtype: int64
Time range: 20224085 to 38310202
Missing values:
student_id    0
skill_id      0
correct       0
time          0
dtype: int64


,student_id,skill_id,correct,time
0,14,2.0,0,21617623
1,14,37.0,0,21617623
2,14,70.0,0,21617623



ASSISTments2017
Rows: 864713
Students: 1709
Skills: 101
Time dtype: int64
Time range: 1095421048 to 1180218370
Missing values:
student_id    0
skill_id      0
correct       0
time          0
dtype: int64


,student_id,skill_id,correct,time
0,8,properties-of-geometric-figures,0,1096470301
1,8,properties-of-geometric-figures,1,1096470350
2,8,sum-of-interior-angles-more-than-3-sides,0,1096470354



Junyi
Rows: 16217311
Students: 72758
Skills: 1326
Time dtype: datetime64[ns, UTC]
Time range: 2018-08-01 07:45:00+00:00 to 2019-08-01 00:00:00+00:00
Missing values:
student_id    0
skill_id      0
correct       0
time          0
dtype: int64


,student_id,skill_id,correct,time
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1,2018-12-22 08:30:00+00:00
1,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1,2018-12-22 08:30:00+00:00
2,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1,2018-12-22 08:30:00+00:00


## Remove interactions with clashing correctness labels for ASSISTments 2017

In [ ]:

# Load only the columns needed to audit ASSISTments 2017 events
assist17_path = DATASETS["ASSISTments2017"]["data_path"]

assist17_audit = pd.read_csv(
    assist17_path,
    usecols=["studentId", "skill", "problemId", "startTime", "correct"],
    low_memory=False
)

assist17_audit = assist17_audit[
    assist17_audit["skill"].notna()
    & (assist17_audit["skill"].astype(str).str.lower() != "noskill")
].copy()

print("Rows after excluding missing skills:", len(assist17_audit))
print("Unique students:", assist17_audit["studentId"].nunique())

event_keys = ["studentId", "problemId", "skill", "startTime"]

# Count distinct correctness labels within each candidate event
label_counts = (
    assist17_audit
    .groupby(event_keys, dropna=False)["correct"]
    .nunique()
)

print("Candidate event groups:", len(label_counts))
print("Repeated event groups:", int((assist17_audit.groupby(
    event_keys, dropna=False
).size() > 1).sum()))
print("Groups with conflicting correctness:", int((label_counts > 1).sum()))
print("Groups with consistent correctness:", int(
    ((label_counts == 1) &
     (assist17_audit.groupby(event_keys, dropna=False).size() > 1)).sum()
))


Rows after excluding missing skills: 864713
Unique students: 1709
Candidate event groups: 858640
Repeated event groups: 5753
Groups with conflicting correctness: 1793
Groups with consistent correctness: 3960


## Save the dataset

In [ ]:

# Clean ASSISTments 2017 event records

event_keys = ["studentId", "problemId", "skill", "startTime"]

# Identify event groups with conflicting correctness labels
label_counts = (
    assist17_audit
    .groupby(event_keys, dropna=False)["correct"]
    .transform("nunique")
)

conflict_rows = label_counts > 1
n_conflict_rows = int(conflict_rows.sum())

# Exclude all rows belonging to conflicting event groups
cleaned = assist17_audit.loc[~conflict_rows].copy()

# Collapse repeated records with the same event identifiers and outcome
before_dedup = len(cleaned)
cleaned = cleaned.drop_duplicates(
    subset=event_keys + ["correct"]
).copy()

print("Rows excluded due to conflicting labels:", n_conflict_rows)
print("Rows before deduplication:", before_dedup)
print("Rows after deduplication:", len(cleaned))
print("Students retained:", cleaned["studentId"].nunique())
print("Skills retained:", cleaned["skill"].nunique())

# Standardize columns for the PFA pipeline
cleaned = cleaned.rename(columns={
    "studentId": "student_id",
    "problemId": "problem_id",
    "skill": "skill_id",
    "startTime": "time",
})

cleaned["correct"] = pd.to_numeric(cleaned["correct"], errors="raise").astype(int)
cleaned["time"] = pd.to_numeric(cleaned["time"], errors="raise")

cleaned = cleaned.sort_values(
    ["student_id", "time", "problem_id", "skill_id"],
    kind="stable"
).reset_index(drop=True)

output_path = os.path.join(
    BASE_DIR,
    "ASSISTments2017",
    "pfa_events_clean.csv"
)
cleaned.to_csv(output_path, index=False)

print("\nSaved cleaned dataset to:", output_path)
display(cleaned.head())


Rows excluded due to conflicting labels: 3734
Rows before deduplication: 860979
Rows after deduplication: 856847
Students retained: 1709
Skills retained: 101

Saved cleaned dataset to: /content/drive/MyDrive/education-ml-research/ASSISTments2017/pfa_events_clean.csv


,student_id,skill_id,problem_id,time,correct
0,8,properties-of-geometric-figures,1118,1096470301,0
1,8,properties-of-geometric-figures,1119,1096470350,1
2,8,sum-of-interior-angles-more-than-3-sides,1120,1096470354,0
3,8,sum-of-interior-angles-more-than-3-sides,1120,1096470360,0
4,8,sum-of-interior-angles-more-than-3-sides,1121,1096470378,1


## Calculate history features

In [ ]:

# Calculate prior success/failure counts for ASSISTments 2017.
# All records at the same student-skill-timestamp share the same history.

pfa17 = pd.read_csv(
    os.path.join(BASE_DIR, "ASSISTments2017", "pfa_events_clean.csv"),
    low_memory=False
)

group_cols = ["student_id", "skill_id", "time"]

# Aggregate outcomes at each student-skill-timestamp.
# This ensures tied timestamps are handled together.
timestamp_counts = (
    pfa17.groupby(group_cols, sort=False)["correct"]
    .agg(
        attempts_at_time="size",
        successes_at_time="sum"
    )
    .reset_index()
)

timestamp_counts["failures_at_time"] = (
    timestamp_counts["attempts_at_time"]
    - timestamp_counts["successes_at_time"]
)

timestamp_counts = timestamp_counts.sort_values(
    ["student_id", "skill_id", "time"],
    kind="stable"
).reset_index(drop=True)

# Cumulative totals minus the current timestamp's totals
skill_groups = timestamp_counts.groupby(
    ["student_id", "skill_id"],
    sort=False
)

timestamp_counts["prior_successes"] = (
    skill_groups["successes_at_time"].cumsum()
    - timestamp_counts["successes_at_time"]
)

timestamp_counts["prior_failures"] = (
    skill_groups["failures_at_time"].cumsum()
    - timestamp_counts["failures_at_time"]
)

# Attach the same prior history to every event at that timestamp
pfa17 = pfa17.merge(
    timestamp_counts[
        group_cols + ["prior_successes", "prior_failures"]
    ],
    on=group_cols,
    how="left",
    validate="many_to_one"
)

print("Rows:", len(pfa17))
print("Missing prior-success counts:", pfa17["prior_successes"].isna().sum())
print("Missing prior-failure counts:", pfa17["prior_failures"].isna().sum())

display(
    pfa17[
        ["student_id", "skill_id", "time", "correct",
         "prior_successes", "prior_failures"]
    ].head(10)
)


Rows: 856847
Missing prior-success counts: 0
Missing prior-failure counts: 0


,student_id,skill_id,time,correct,prior_successes,prior_failures
0,8,properties-of-geometric-figures,1096470301,0,0,0
1,8,properties-of-geometric-figures,1096470350,1,0,1
2,8,sum-of-interior-angles-more-than-3-sides,1096470354,0,0,0
3,8,sum-of-interior-angles-more-than-3-sides,1096470360,0,0,1
4,8,sum-of-interior-angles-more-than-3-sides,1096470378,1,0,2
5,8,sum-of-interior-angles-more-than-3-sides,1096470380,0,1,2
6,8,sum-of-interior-angles-more-than-3-sides,1096470439,1,1,3
7,8,point-plotting,1096470454,0,0,0
8,8,transformations-rotations,1096470515,0,0,0
9,8,transformations-rotations,1096470547,1,0,1


## Merge student data with groups

In [ ]:

# Attach independently estimated IRT ability groups
groups_path = DATASETS["ASSISTments2017"]["groups_path"]
ability17 = pd.read_csv(groups_path)

print("Ability file columns:", ability17.columns.tolist())
display(ability17.head())

# Use the student identifier and existing IRT group labels
pfa17["student_id"] = pfa17["student_id"].astype(str)
ability17["studentId"] = ability17["studentId"].astype(str)

pfa17 = pfa17.merge(
    ability17[["studentId", "theta_quartile", "theta_sd_group"]],
    left_on="student_id",
    right_on="studentId",
    how="left",
    validate="many_to_one"
).drop(columns=["studentId"])

print("\nRows:", len(pfa17))
print("Rows missing IRT quartile:", pfa17["theta_quartile"].isna().sum())
print("Rows missing IRT SD group:", pfa17["theta_sd_group"].isna().sum())

print("\nStudents per IRT quartile:")
print(
    ability17["theta_quartile"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nStudents per IRT SD group:")
print(ability17["theta_sd_group"].value_counts(dropna=False))


Ability file columns: ['studentId', 'theta_all', 'theta_first', 'theta_quartile', 'theta_sd_group']


,studentId,theta_all,theta_first,theta_quartile,theta_sd_group
0,8,0.145408,0.659446,Q4,0 to +0.5 SD
1,9,0.040027,0.334709,Q3,0 to +0.5 SD
2,11,-0.626556,-0.538717,Q2,-1.5 to -1.0 SD
3,25,0.339209,0.544214,Q4,+0.5 to +1.0 SD
4,27,-0.264493,0.360488,Q3,-0.5 to 0 SD



Rows: 856847
Rows missing IRT quartile: 0
Rows missing IRT SD group: 0

Students per IRT quartile:
theta_quartile
Q1    428
Q2    427
Q3    427
Q4    427
Name: count, dtype: int64

Students per IRT SD group:
theta_sd_group
-1.0 to -0.5 SD    407
-1.5 to -1.0 SD    330
-0.5 to 0 SD       314
0 to +0.5 SD       232
< -1.5 SD          136
+0.5 to +1.0 SD    134
+1.0 to +1.5 SD     82
> +1.5 SD           74
Name: count, dtype: int64


## Create a student-level train/test split

In [ ]:

from sklearn.model_selection import train_test_split

# One row per student, preserving the saved ability groups
student_groups = ability17[
    ["studentId", "theta_quartile", "theta_sd_group"]
].copy()

student_groups["studentId"] = student_groups["studentId"].astype(str)

train_students, test_students = train_test_split(
    student_groups["studentId"].unique(),
    test_size=0.20,
    random_state=SEED
)

train_students = set(train_students)
test_students = set(test_students)

train_df = pfa17[pfa17["student_id"].isin(train_students)].copy()
test_df = pfa17[pfa17["student_id"].isin(test_students)].copy()

print("Training students:", len(train_students))
print("Test students:", len(test_students))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Student overlap:", len(train_students & test_students))

print("\nTest-set students by quartile:")
print(
    student_groups[
        student_groups["studentId"].isin(test_students)
    ]["theta_quartile"].value_counts().sort_index()
)

print("\nTest-set students by SD group:")
print(
    student_groups[
        student_groups["studentId"].isin(test_students)
    ]["theta_sd_group"].value_counts()
)


Training students: 1367
Test students: 342
Training rows: 679214
Test rows: 177633
Student overlap: 0

Test-set students by quartile:
theta_quartile
Q1    95
Q2    73
Q3    94
Q4    80
Name: count, dtype: int64

Test-set students by SD group:
theta_sd_group
-1.0 to -0.5 SD    74
-0.5 to 0 SD       71
-1.5 to -1.0 SD    66
0 to +0.5 SD       42
< -1.5 SD          32
+0.5 to +1.0 SD    27
+1.0 to +1.5 SD    16
> +1.5 SD          14
Name: count, dtype: int64


## Define and validate classical PFA

In [ ]:

import numpy as np
import pandas as pd

# Synthetic example: one student practicing one skill.
# The last two interactions share a timestamp.
example = pd.DataFrame({
    "student_id": ["A", "A", "A", "A"],
    "skill_id":   ["S1", "S1", "S1", "S1"],
    "time":       [1, 2, 3, 3],
    "correct":    [1, 0, 1, 0],
})

def add_pfa_history(df):
    """Count prior successes and failures per student and skill.
    Interactions tied at the same timestamp share the same history.
    """
    df = df.copy()
    df["_row_order"] = np.arange(len(df))

    # Aggregate outcomes at each student-skill-timestamp.
    keys = ["student_id", "skill_id", "time"]
    grouped = (
        df.groupby(keys, as_index=False, sort=False)
          .agg(
              attempts_at_time=("correct", "size"),
              successes_at_time=("correct", "sum"),
          )
    )
    grouped["failures_at_time"] = (
        grouped["attempts_at_time"] - grouped["successes_at_time"]
    )

    grouped = grouped.sort_values(keys, kind="stable")

    # Cumulative totals minus current timestamp's outcomes
    # ensure no interaction sees another interaction at the same time.
    grouped["prior_successes"] = (
        grouped.groupby(["student_id", "skill_id"])["successes_at_time"]
        .cumsum() - grouped["successes_at_time"]
    )
    grouped["prior_failures"] = (
        grouped.groupby(["student_id", "skill_id"])["failures_at_time"]
        .cumsum() - grouped["failures_at_time"]
    )

    df = df.merge(
        grouped[keys + ["prior_successes", "prior_failures"]],
        on=keys,
        how="left",
        validate="many_to_one",
    )

    return (
        df.sort_values("_row_order")
          .drop(columns="_row_order")
          .reset_index(drop=True)
    )

example_with_history = add_pfa_history(example)
print(example_with_history)


  student_id skill_id  time  correct  prior_successes  prior_failures
0          A       S1     1        1                0               0
1          A       S1     2        0                1               0
2          A       S1     3        1                1               1
3          A       S1     3        0                1               1


## Fit the classical PFA model

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy import sparse

# Expand the synthetic data so the model has enough observations
# to demonstrate fitting. This is only a software test.
rng = np.random.default_rng(42)
n = 500

test_data = pd.DataFrame({
    "student_id": rng.choice([f"S{i}" for i in range(30)], size=n),
    "skill_id": rng.choice(["K1", "K2", "K3"], size=n),
    "time": np.tile(np.arange(n // 30 + 1), 30)[:n],
})
test_data = test_data.sort_values(
    ["student_id", "time"], kind="stable"
).reset_index(drop=True)

# Generate illustrative outcomes for the software test.
# Outcomes are simulated and are NOT research results.
test_data["correct"] = rng.binomial(1, 0.6, size=len(test_data))

test_data = add_pfa_history(test_data)

# Create skill-specific feature blocks:
# [skill indicators | successes by skill | failures by skill]
skills = sorted(test_data["skill_id"].unique())
skill_to_idx = {skill: idx for idx, skill in enumerate(skills)}
skill_idx = test_data["skill_id"].map(skill_to_idx).to_numpy()
n_skills = len(skills)
n_rows = len(test_data)

X_skill = np.zeros((n_rows, n_skills))
X_success = np.zeros((n_rows, n_skills))
X_failure = np.zeros((n_rows, n_skills))

row_idx = np.arange(n_rows)
X_skill[row_idx, skill_idx] = 1
X_success[row_idx, skill_idx] = test_data["prior_successes"].to_numpy()
X_failure[row_idx, skill_idx] = test_data["prior_failures"].to_numpy()

X = np.hstack([X_skill, X_success, X_failure])
y = test_data["correct"].to_numpy()

pfa_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000
)
pfa_model.fit(X, y)

probabilities = pfa_model.predict_proba(X)[:, 1]

print("Skills:", skills)
print("Feature matrix shape:", X.shape)
print("Intercepts:", pfa_model.intercept_)
print("Coefficient vector shape:", pfa_model.coef_.shape)
print("Training AUC (software check only):", roc_auc_score(y, probabilities))
print("Training Brier score (software check only):", brier_score_loss(y, probabilities))


Skills: ['K1', 'K2', 'K3']
Feature matrix shape: (500, 9)
Intercepts: [0.32857985]
Coefficient vector shape: (1, 9)
Training AUC (software check only): 0.5348042418455042
Training Brier score (software check only): 0.23729041080180413


## Ensure each skill has its own intercept

In [ ]:

# Refit classical PFA with one intercept per skill
pfa_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000,
    fit_intercept=False
)
pfa_model.fit(X, y)

coefficients = pfa_model.coef_[0]

pfa_parameters = pd.DataFrame({
    "skill_id": skills,
    "beta_intercept": coefficients[:n_skills],
    "gamma_success": coefficients[n_skills:2*n_skills],
    "rho_failure": coefficients[2*n_skills:3*n_skills],
})

print(pfa_parameters.to_string(index=False))

probabilities = pfa_model.predict_proba(X)[:, 1]
print("\nFeature matrix shape:", X.shape)
print("Coefficient count:", len(coefficients))
print("Training AUC (software check only):", roc_auc_score(y, probabilities))
print("Training Brier score (software check only):", brier_score_loss(y, probabilities))


skill_id  beta_intercept  gamma_success  rho_failure
      K1        0.585008      -0.129911     0.010908
      K2        0.487113      -0.024938     0.097290
      K3        0.242818       0.013816     0.072297

Feature matrix shape: (500, 9)
Coefficient count: 9
Training AUC (software check only): 0.5352733242867432
Training Brier score (software check only): 0.23729034292344764


## Run PFA on ASSISTments 2017

In [ ]:

from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

# Check that the cleaned dataset and student-level split are available.
required_cols = [
    "student_id", "skill_id", "correct",
    "prior_successes", "prior_failures",
    "theta_quartile", "theta_sd_group"
]

for name, frame in [("pfa17", pfa17), ("train_df", train_df), ("test_df", test_df)]:
    missing = set(required_cols) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

assert set(train_df["student_id"]).isdisjoint(set(test_df["student_id"])), \
    "Train/test student overlap detected."

# Use a fixed skill-to-column mapping for both train and test.
real_skills = sorted(pfa17["skill_id"].unique())
skill_to_idx = {skill: i for i, skill in enumerate(real_skills)}
n_skills = len(real_skills)

def make_pfa_matrix(frame, skill_to_idx, n_skills):
    """Sparse classical PFA features:
    skill intercept indicators, prior successes, prior failures.
    """
    skill_idx = frame["skill_id"].map(skill_to_idx)

    if skill_idx.isna().any():
        raise ValueError("Found a skill without a feature index.")

    skill_idx = skill_idx.to_numpy(dtype=np.int32)
    row_idx = np.arange(len(frame), dtype=np.int32)

    success = frame["prior_successes"].to_numpy(dtype=np.float64)
    failure = frame["prior_failures"].to_numpy(dtype=np.float64)

    rows = np.concatenate([row_idx, row_idx, row_idx])
    cols = np.concatenate([
        skill_idx,
        n_skills + skill_idx,
        2 * n_skills + skill_idx
    ])
    values = np.concatenate([
        np.ones(len(frame), dtype=np.float64),
        success,
        failure
    ])

    return sparse.coo_matrix(
        (values, (rows, cols)),
        shape=(len(frame), 3 * n_skills)
    ).tocsr()

X_train = make_pfa_matrix(train_df, skill_to_idx, n_skills)
X_test = make_pfa_matrix(test_df, skill_to_idx, n_skills)

y_train = train_df["correct"].to_numpy(dtype=np.int32)
y_test = test_df["correct"].to_numpy(dtype=np.int32)

print("Training matrix:", X_train.shape, "| nonzero values:", X_train.nnz)
print("Test matrix:", X_test.shape, "| nonzero values:", X_test.nnz)
print("Number of skills:", n_skills)
print("Training students:", train_df["student_id"].nunique())
print("Test students:", test_df["student_id"].nunique())
print("Student overlap:", len(set(train_df["student_id"]) & set(test_df["student_id"])))


Training matrix: (679214, 303) | nonzero values: 2037642
Test matrix: (177633, 303) | nonzero values: 532899
Number of skills: 101
Training students: 1367
Test students: 342
Student overlap: 0


## Fit classical PFA

In [ ]:

# Fit classical PFA on the training students only.
pfa_model_17 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
    fit_intercept=False,
    tol=1e-4
)

pfa_model_17.fit(X_train, y_train)

# Predict responses for held-out students.
pfa_prob_17 = pfa_model_17.predict_proba(X_test)[:, 1]

# Evaluate overall held-out predictive performance.
pfa_auc_17 = roc_auc_score(y_test, pfa_prob_17)
pfa_brier_17 = brier_score_loss(y_test, pfa_prob_17)

print("Converged:", pfa_model_17.n_iter_[0] < pfa_model_17.max_iter)
print("Iterations:", pfa_model_17.n_iter_[0])
print("Test AUC:", round(pfa_auc_17, 6))
print("Test Brier score:", round(pfa_brier_17, 6))
print("Predicted probabilities range:",
      round(pfa_prob_17.min(), 6), "to", round(pfa_prob_17.max(), 6))


Converged: True
Iterations: 929
Test AUC: 0.614416
Test Brier score: 0.221434
Predicted probabilities range: 0.023593 to 0.971064


## Evaluate PFA by IRT ability group

In [ ]:

# Attach the held-out predictions to the corresponding test rows.
pfa_eval_17 = test_df[
    ["student_id", "correct", "theta_quartile", "theta_sd_group"]
].copy().reset_index(drop=True)

assert len(pfa_eval_17) == len(pfa_prob_17), "Prediction/row count mismatch"

pfa_eval_17["probability"] = pfa_prob_17

def summarize_pfa_groups(df, group_col):
    results = []

    for group, part in df.groupby(group_col, observed=True, dropna=False):
        y_true = part["correct"].to_numpy()
        y_prob = part["probability"].to_numpy()

        auc = (
            roc_auc_score(y_true, y_prob)
            if len(np.unique(y_true)) == 2
            else np.nan
        )

        results.append({
            "ability_group": group,
            "n_responses": len(part),
            "n_students": part["student_id"].nunique(),
            "AUC": auc,
            "Brier": brier_score_loss(y_true, y_prob),
            "correct_rate": y_true.mean()
        })

    return pd.DataFrame(results)

quartile_results_17 = summarize_pfa_groups(
    pfa_eval_17, "theta_quartile"
)

sd_results_17 = summarize_pfa_groups(
    pfa_eval_17, "theta_sd_group"
)

print("PFA results by IRT quartile:")
display(quartile_results_17.round(4))

print("PFA results by IRT standard-deviation band:")
display(sd_results_17.round(4))


PFA results by IRT quartile:


,ability_group,n_responses,n_students,AUC,Brier,correct_rate
0,Q1,58892,95,0.5514,0.2018,0.2651
1,Q2,41659,73,0.5938,0.2145,0.3220
2,Q3,47098,94,0.6184,0.2299,0.4003
3,Q4,29984,80,0.6149,0.2562,0.5379


PFA results by IRT standard-deviation band:


,ability_group,n_responses,n_students,AUC,Brier,correct_rate
0,+0.5 to +1.0 SD,11655,27,0.6145,0.2523,0.5171
1,+1.0 to +1.5 SD,4499,16,0.6115,0.2640,0.5826
2,-0.5 to 0 SD,37505,71,0.6198,0.2283,0.3935
3,-1.0 to -0.5 SD,41100,74,0.5945,0.2156,0.3261
4,-1.5 to -1.0 SD,41265,66,0.5616,0.2060,0.2811
5,0 to +0.5 SD,17821,42,0.6114,0.2411,0.4538
6,< -1.5 SD,19846,32,0.5236,0.1934,0.2335
7,> +1.5 SD,3942,14,0.5810,0.2916,0.7288


## Compare against baseline

In [ ]:

# Baseline: predict the overall training-set correct-response rate
train_correct_rate = y_train.mean()
baseline_prob_17 = np.full(len(y_test), train_correct_rate)

print("Training-set correct rate:", round(train_correct_rate, 4))
print("Overall baseline Brier score:",
      round(brier_score_loss(y_test, baseline_prob_17), 6))
print("Overall PFA Brier score:",
      round(brier_score_loss(y_test, pfa_prob_17), 6))

# Compare PFA with the same baseline within each ability group.
baseline_eval = pfa_eval_17.copy()
baseline_eval["baseline_probability"] = train_correct_rate

baseline_rows = []
for group, part in baseline_eval.groupby("theta_quartile", observed=True):
    y_true = part["correct"].to_numpy()

    baseline_rows.append({
        "ability_group": group,
        "n_students": part["student_id"].nunique(),
        "correct_rate": y_true.mean(),
        "PFA_Brier": brier_score_loss(
            y_true, part["probability"].to_numpy()
        ),
        "Baseline_Brier": brier_score_loss(
            y_true, part["baseline_probability"].to_numpy()
        ),
    })

baseline_comparison_17 = pd.DataFrame(baseline_rows)
baseline_comparison_17["Brier_improvement"] = (
    baseline_comparison_17["Baseline_Brier"]
    - baseline_comparison_17["PFA_Brier"]
)

display(baseline_comparison_17.round(4))


Training-set correct rate: 0.3707
Overall baseline Brier score: 0.230595
Overall PFA Brier score: 0.221434


,ability_group,n_students,correct_rate,PFA_Brier,Baseline_Brier,Brier_improvement
0,Q1,95,0.2651,0.2018,0.2060,0.0041
1,Q2,73,0.3220,0.2145,0.2207,0.0062
2,Q3,94,0.4003,0.2299,0.2409,0.0110
3,Q4,80,0.5379,0.2562,0.2765,0.0203


## PFA Results on ASSISTments 2017

We implemented classical PFA using skill-specific intercepts and counts of prior correct and incorrect responses. The model was evaluated using a student-level train/test split.

### Overall Performance

| Metric | PFA |
|---|---:|
| Test AUC | 0.614416 |
| Test Brier Score | 0.221434 |

### IRT Quartiles

| Ability Group | Test Responses | Test Students | AUC | Brier Score |
|---|---:|---:|---:|---:|
| Q1 (Lowest) | 58,892 | 95 | 0.5514 | 0.2018 |
| Q2 | 41,659 | 73 | 0.5938 | 0.2145 |
| Q3 | 47,098 | 94 | 0.6184 | 0.2299 |
| Q4 (Highest) | 29,984 | 80 | 0.6149 | 0.2562 |

### IRT Standard-Deviation Groups

| Ability Group | Test Responses | Test Students | AUC | Brier Score |
|---|---:|---:|---:|---:|
| Below −1.5 SD | 19,846 | 32 | 0.5236 | 0.1934 |
| −1.5 to −1.0 SD | 41,265 | 66 | 0.5616 | 0.2060 |
| −1.0 to −0.5 SD | 41,100 | 74 | 0.5945 | 0.2156 |
| −0.5 to 0 SD | 37,505 | 71 | 0.6198 | 0.2283 |
| 0 to +0.5 SD | 17,821 | 42 | 0.6114 | 0.2411 |
| +0.5 to +1.0 SD | 11,655 | 27 | 0.6145 | 0.2523 |
| +1.0 to +1.5 SD | 4,499 | 16 | 0.6115 | 0.2640 |
| Above +1.5 SD | 3,942 | 14 | 0.5810 | 0.2916 |



## Load ASSISTments 2009 for PFA preprocessing

In [ ]:
# Load ASSISTments 2009 for PFA preprocessing

assist09_path = DATASETS["ASSISTments2009"]["data_path"]
ability09_path = DATASETS["ASSISTments2009"]["groups_path"]

assist09_raw = pd.read_csv(
    assist09_path,
    encoding="latin-1",
    low_memory=False
)

ability09 = pd.read_csv(ability09_path)

print("Dataset shape:", assist09_raw.shape)
print("\nColumns:")
print(assist09_raw.columns.tolist())

print("\nAbility-group file shape:", ability09.shape)
print("\nFirst five rows:")
display(assist09_raw.head())

print("\nMissing values in key columns:")
key_cols = [
    c for c in ["order_id", "user_id", "problem_id",
                "skill_id", "correct", "attempt_count"]
    if c in assist09_raw.columns
]
display(assist09_raw[key_cols].isna().sum().to_frame("missing_count"))

Dataset shape: (525534, 30)

Columns:
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']

Ability-group file shape: (4163, 4)

First five rows:


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0



Missing values in key columns:


,missing_count
order_id,0
user_id,0
problem_id,0
skill_id,66326
correct,0
attempt_count,0


## Preprocess ASSISTments 2009 for classical PFA

In [ ]:
assist09_pfa = assist09_raw[
    ["user_id", "skill_id", "correct", "order_id", "problem_id"]
].copy()

# Remove rows missing essential fields.
assist09_pfa = assist09_pfa.dropna(
    subset=["user_id", "skill_id", "correct", "order_id"]
).copy()

# Standardize types.
assist09_pfa["student_id"] = assist09_pfa["user_id"].astype(str)
assist09_pfa["skill_id"] = assist09_pfa["skill_id"].astype(str)
assist09_pfa["time"] = pd.to_numeric(
    assist09_pfa["order_id"], errors="raise"
)
assist09_pfa["correct"] = pd.to_numeric(
    assist09_pfa["correct"], errors="raise"
)

# Retain only valid binary response labels.
assist09_pfa = assist09_pfa[
    assist09_pfa["correct"].isin([0, 1])
].copy()
assist09_pfa["correct"] = assist09_pfa["correct"].astype(np.int8)

# Sort chronologically within each student.
assist09_pfa = (
    assist09_pfa[
        ["student_id", "skill_id", "time", "problem_id", "correct"]
    ]
    .sort_values(["student_id", "time", "problem_id", "skill_id"],
                 kind="stable", na_position="last")
    .reset_index(drop=True)
)

print("Rows:", len(assist09_pfa))
print("Students:", assist09_pfa["student_id"].nunique())
print("Skills:", assist09_pfa["skill_id"].nunique())
print("Missing required values:")
display(assist09_pfa[["student_id", "skill_id", "time", "correct"]].isna().sum())
print("\nSample:")
display(assist09_pfa.head())

Rows: 459208
Students: 4163
Skills: 123
Missing required values:


,0
student_id,0
skill_id,0
time,0
correct,0



Sample:


,student_id,skill_id,time,problem_id,correct
0,14,2.0,21617623,93383,0
1,14,37.0,21617623,93383,0
2,14,70.0,21617623,93383,0
3,14,2.0,21617632,93407,1
4,14,37.0,21617632,93407,1


## Calculate PFA history features

In [ ]:
# Calculate prior successes and failures for PFA
assist09_pfa = add_pfa_history(assist09_pfa)

print("Rows:", len(assist09_pfa))
print("Missing history values:")
display(
    assist09_pfa[["prior_successes", "prior_failures"]]
    .isna()
    .sum()
)

print("\nHistory feature summary:")
display(
    assist09_pfa[["prior_successes", "prior_failures"]]
    .describe()
    .round(2)
)

print("\nExample student history:")
display(
    assist09_pfa.loc[
        assist09_pfa["student_id"] == "14",
        ["student_id", "skill_id", "time", "correct",
         "prior_successes", "prior_failures"]
    ].head(12)
)

Rows: 459208
Missing history values:


,0
prior_successes,0
prior_failures,0



History feature summary:


,prior_successes,prior_failures
count,459208.00,459208.00
mean,43.39,24.22
std,178.15,105.29
min,0.00,0.00
25%,0.00,0.00
50%,3.00,1.00
75%,12.00,6.00
max,2543.00,1260.00



Example student history:


,student_id,skill_id,time,correct,prior_successes,prior_failures
0,14,2.0,21617623,0,0,0
1,14,37.0,21617623,0,0,0
2,14,70.0,21617623,0,0,0
3,14,2.0,21617632,1,0,1
4,14,37.0,21617632,1,0,1
5,14,70.0,21617632,1,0,1
6,14,2.0,21617641,0,1,1
7,14,37.0,21617641,0,1,1
8,14,70.0,21617641,0,1,1
9,14,2.0,21617650,0,1,2


## Merge IRT groups with dataset

In [ ]:
# Standardize student IDs before merging
assist09_pfa["student_id"] = assist09_pfa["student_id"].astype(str)
ability09_merge = ability09.copy()
ability09_merge["user_id"] = ability09_merge["user_id"].astype(str)

assist09_pfa = assist09_pfa.merge(
    ability09_merge[
        ["user_id", "theta", "theta_quartile", "theta_sd_group"]
    ],
    left_on="student_id",
    right_on="user_id",
    how="left",
    validate="many_to_one"
).drop(columns="user_id")

print("Rows:", len(assist09_pfa))
print("Students:", assist09_pfa["student_id"].nunique())
print("Missing ability groups:")
display(
    assist09_pfa[["theta", "theta_quartile", "theta_sd_group"]]
    .isna()
    .sum()
)

print("\nQuartile student counts:")
display(
    ability09_merge["theta_quartile"].value_counts().sort_index()
)

print("\nSD-group student counts:")
display(
    ability09_merge["theta_sd_group"].value_counts()
)

Rows: 459208
Students: 4163
Missing ability groups:


,0
theta,0
theta_quartile,0
theta_sd_group,0



Quartile student counts:


,count
theta_quartile,
Q1,1041
Q2,1041
Q3,1040
Q4,1041



SD-group student counts:


,count
theta_sd_group,
-0.5 to 0 SD,1129
0 to +0.5 SD,933
-1.0 to -0.5 SD,704
+0.5 to +1.0 SD,381
< -1.5 SD,351
-1.5 to -1.0 SD,299
> +1.5 SD,216
+1.0 to +1.5 SD,150


## Create train/test split

In [ ]:
from sklearn.model_selection import train_test_split

assist09_students = assist09_pfa["student_id"].unique()

assist09_train_students, assist09_test_students = train_test_split(
    assist09_students,
    test_size=0.20,
    random_state=SEED
)

assist09_train = assist09_pfa[
    assist09_pfa["student_id"].isin(assist09_train_students)
].copy()

assist09_test = assist09_pfa[
    assist09_pfa["student_id"].isin(assist09_test_students)
].copy()

print("Training students:", len(assist09_train_students))
print("Test students:", len(assist09_test_students))
print("Training rows:", len(assist09_train))
print("Test rows:", len(assist09_test))

print(
    "Student overlap:",
    len(set(assist09_train_students) & set(assist09_test_students))
)

Training students: 3330
Test students: 833
Training rows: 361682
Test rows: 97526
Student overlap: 0


## Create PFA to match ASSISTments 2009

In [ ]:
from scipy import sparse
import numpy as np

# Create a consistent skill-to-index mapping
assist09_skills = sorted(assist09_pfa["skill_id"].unique())
assist09_skill_to_idx = {
    skill: i for i, skill in enumerate(assist09_skills)
}
assist09_n_skills = len(assist09_skills)

def make_pfa_matrix(frame, skill_to_idx, n_skills):
    skill_idx = frame["skill_id"].map(skill_to_idx)

    if skill_idx.isna().any():
        raise ValueError("Found skills missing from the skill mapping.")

    skill_idx = skill_idx.to_numpy(dtype=np.int32)
    row_idx = np.arange(len(frame), dtype=np.int32)

    success = frame["prior_successes"].to_numpy(dtype=np.float64)
    failure = frame["prior_failures"].to_numpy(dtype=np.float64)

    rows = np.concatenate([row_idx, row_idx, row_idx])
    cols = np.concatenate([
        skill_idx,
        n_skills + skill_idx,
        2 * n_skills + skill_idx
    ])
    values = np.concatenate([
        np.ones(len(frame), dtype=np.float64),
        success,
        failure
    ])

    return sparse.coo_matrix(
        (values, (rows, cols)),
        shape=(len(frame), 3 * n_skills)
    ).tocsr()

X_train_09 = make_pfa_matrix(
    assist09_train, assist09_skill_to_idx, assist09_n_skills
)
X_test_09 = make_pfa_matrix(
    assist09_test, assist09_skill_to_idx, assist09_n_skills
)

y_train_09 = assist09_train["correct"].to_numpy(dtype=np.int8)
y_test_09 = assist09_test["correct"].to_numpy(dtype=np.int8)

print("Number of skills:", assist09_n_skills)
print("Training matrix:", X_train_09.shape)
print("Test matrix:", X_test_09.shape)
print("Training nonzero entries:", X_train_09.nnz)
print("Test nonzero entries:", X_test_09.nnz)

Number of skills: 123
Training matrix: (361682, 369)
Test matrix: (97526, 369)
Training nonzero entries: 1085046
Test nonzero entries: 292578


## Fit the model

In [ ]:
from scipy import sparse

# Keep skill-intercept features unchanged.
# Scale prior success/failure counts by 1/100.
scale_factors = np.concatenate([
    np.ones(assist09_n_skills),
    np.full(2 * assist09_n_skills, 0.01)
])

X_train_09_scaled = X_train_09 @ sparse.diags(scale_factors)
X_test_09_scaled = X_test_09 @ sparse.diags(scale_factors)

pfa_model_09 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    fit_intercept=False,
    tol=1e-4
)

pfa_model_09.fit(X_train_09_scaled, y_train_09)

pfa_prob_09 = pfa_model_09.predict_proba(X_test_09_scaled)[:, 1]

print("Iterations:", pfa_model_09.n_iter_[0])
print("Test AUC:", round(roc_auc_score(y_test_09, pfa_prob_09), 6))
print("Test Brier Score:", round(brier_score_loss(y_test_09, pfa_prob_09), 6))

Iterations: 484
Test AUC: 0.6937
Test Brier Score: 0.188537


## Test the model

In [ ]:
# Prepare held-out test results
assist09_test_results = assist09_test[
    ["student_id", "theta_quartile", "theta_sd_group", "correct"]
].copy()

assist09_test_results["predicted_prob"] = pfa_prob_09


def evaluate_pfa_groups(df, group_col, group_order):
    results = []

    for group in group_order:
        subset = df[df[group_col] == group]

        results.append({
            "Ability group": group,
            "Responses": len(subset),
            "Students": subset["student_id"].nunique(),
            "AUC": (
                roc_auc_score(subset["correct"], subset["predicted_prob"])
                if subset["correct"].nunique() == 2 else np.nan
            ),
            "Brier Score": brier_score_loss(
                subset["correct"], subset["predicted_prob"]
            ),
            "Correct rate": subset["correct"].mean()
        })

    return pd.DataFrame(results)


# Quartile results
quartile_results_09 = evaluate_pfa_groups(
    assist09_test_results,
    "theta_quartile",
    ["Q1", "Q2", "Q3", "Q4"]
)

# Standard-deviation group results
sd_order_09 = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

sd_results_09 = evaluate_pfa_groups(
    assist09_test_results,
    "theta_sd_group",
    sd_order_09
)

print("ASSISTments 2009 — PFA by IRT Quartile")
display(quartile_results_09.round(4))

print("\nASSISTments 2009 — PFA by IRT Standard-Deviation Group")
display(sd_results_09.round(4))

ASSISTments 2009 — PFA by IRT Quartile


,Ability group,Responses,Students,AUC,Brier Score,Correct rate
0,Q1,10161,219,0.6680,0.2743,0.3386
1,Q2,24569,206,0.6722,0.2230,0.6039
2,Q3,42450,217,0.6494,0.1774,0.7570
3,Q4,20346,191,0.6723,0.1273,0.8737



ASSISTments 2009 — PFA by IRT Standard-Deviation Group


,Ability group,Responses,Students,AUC,Brier Score,Correct rate
0,< -1.5 SD,2274,81,0.6593,0.2574,0.0888
1,-1.5 to -1.0 SD,2327,53,0.6703,0.2716,0.2884
2,-1.0 to -0.5 SD,12543,149,0.6105,0.2635,0.5178
3,-0.5 to 0 SD,28980,230,0.6718,0.2102,0.6450
4,0 to +0.5 SD,40655,176,0.6464,0.1631,0.7926
5,+0.5 to +1.0 SD,5207,67,0.6296,0.1353,0.8761
6,+1.0 to +1.5 SD,3802,36,0.7035,0.0836,0.9516
7,> +1.5 SD,1738,41,0.2731,0.0689,0.9942


# ASSISTments 2009 — PFA Results

## Overall Test Performance

| Metric | Result |
|---|---:|
| Test AUC | 0.693700 |
| Test Brier Score | 0.188537 |
| Convergence Iterations | 484 |
| Training Students | 3,330 |
| Test Students | 833 |
| Training Responses | 361,682 |
| Test Responses | 97,526 |
| Student Overlap | 0 |

## PFA by IRT Ability Quartile

| Ability Group | Responses | Students | AUC | Brier Score | Correct Rate |
|---|---:|---:|---:|---:|---:|
| Q1 | 10,161 | 219 | 0.6680 | 0.2743 | 0.3386 |
| Q2 | 24,569 | 206 | 0.6722 | 0.2230 | 0.6039 |
| Q3 | 42,450 | 217 | 0.6494 | 0.1774 | 0.7570 |
| Q4 | 20,346 | 191 | 0.6723 | 0.1273 | 0.8737 |

## PFA by IRT Standard-Deviation Group

| Ability Group | Responses | Students | AUC | Brier Score | Correct Rate |
|---|---:|---:|---:|---:|---:|
| < -1.5 SD | 2,274 | 81 | 0.6593 | 0.2574 | 0.0888 |
| -1.5 to -1.0 SD | 2,327 | 53 | 0.6703 | 0.2716 | 0.2884 |
| -1.0 to -0.5 SD | 12,543 | 149 | 0.6105 | 0.2635 | 0.5178 |
| -0.5 to 0 SD | 28,980 | 230 | 0.6718 | 0.2102 | 0.6450 |
| 0 to +0.5 SD | 40,655 | 176 | 0.6464 | 0.1631 | 0.7926 |
| +0.5 to +1.0 SD | 5,207 | 67 | 0.6296 | 0.1353 | 0.8761 |
| +1.0 to +1.5 SD | 3,802 | 36 | 0.7035 | 0.0836 | 0.9516 |
| > +1.5 SD | 1,738 | 41 | 0.2731 | 0.0689 | 0.9942 |

## Methodological Notes

- **Model:** Classical Performance Factors Analysis (PFA), implemented using unregularized logistic regression.
- **Features:** Skill-specific intercepts, prior successful attempts, and prior unsuccessful attempts.
- **Optimization:** Prior-success and prior-failure features were scaled by 0.01 to improve convergence. The model converged after 484 iterations.
- **Data split:** Student-level 80/20 train/test split with random seed 42; no students appeared in both sets.
- **Ability grouping:** Students were grouped using IRT ability estimates ($\theta$), with quartile and standard-deviation groupings evaluated separately.
- **Interpretation caution:** The > +1.5 SD group had a 99.42% correct rate, so its AUC of 0.2731 may be unstable because very few responses were incorrect.
- **Evaluation status:** These are preliminary results from the current test split, which has been inspected during model development.

## Merge IRT groups for Junyi

In [ ]:
ability_junyi = pd.read_csv(
    os.path.join(
        BASE_DIR,
        "Junyi",
        "student_irt_ability_groups.csv"
    )
)

print("Shape:", ability_junyi.shape)
print("Columns:", ability_junyi.columns.tolist())
display(ability_junyi.head())

junyi_pfa = pfa_data["Junyi"].copy()

# Standardize identifiers for the merge
junyi_pfa["student_id"] = junyi_pfa["student_id"].astype(str)
ability_junyi["uuid"] = ability_junyi["uuid"].astype(str)

junyi_pfa = junyi_pfa.merge(
    ability_junyi[
        ["uuid", "theta_all", "theta_quartile", "theta_sd_group"]
    ],
    left_on="student_id",
    right_on="uuid",
    how="left",
    validate="many_to_one"
).drop(columns="uuid")

print("Rows:", len(junyi_pfa))
print("Students:", junyi_pfa["student_id"].nunique())

print("\nMissing values:")
display(
    junyi_pfa[
        ["theta_all", "theta_quartile", "theta_sd_group"]
    ].isna().sum()
)

print("\nQuartile student counts:")
display(ability_junyi["theta_quartile"].value_counts().sort_index())

Shape: (72758, 4)
Columns: ['uuid', 'theta_all', 'theta_quartile', 'theta_sd_group']


,uuid,theta_all,theta_quartile,theta_sd_group
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,3.095891,Q4,> +1.5 SD
1,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,-2.329677,Q1,< -1.5 SD
2,++E4TrlDYvGtPBg1edhkLXLEEbnfiAgAamPQ33vpW8M=,-0.632970,Q2,-1.0 to -0.5 SD
3,++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=,-0.265644,Q2,-0.5 to 0 SD
4,++GobOSWqrsaxoRg1bMN+T6biIJcgBXwuOH/ddq3DiU=,0.577118,Q3,+0.5 to +1.0 SD


Rows: 16217311
Students: 72758

Missing values:


,0
theta_all,0
theta_quartile,0
theta_sd_group,0



Quartile student counts:


,count
theta_quartile,
Q1,18190
Q2,18189
Q3,18189
Q4,18190


## Create train/test split

In [ ]:
from sklearn.model_selection import train_test_split

junyi_students = ability_junyi["uuid"].astype(str).to_numpy()

junyi_train_students, junyi_test_students = train_test_split(
    junyi_students,
    test_size=0.20,
    random_state=SEED
)

junyi_train_set = set(junyi_train_students)
junyi_test_set = set(junyi_test_students)

print("Training students:", len(junyi_train_set))
print("Test students:", len(junyi_test_set))
print(
    "Student overlap:",
    len(junyi_train_set & junyi_test_set)
)

Training students: 58206
Test students: 14552
Student overlap: 0


In [ ]:
del pfa_data["Junyi"]
import gc
gc.collect()

print("Merged Junyi data:", junyi_pfa.shape)
print("Ability columns present:", [
    col for col in ["theta_all", "theta_quartile", "theta_sd_group"]
    if col in junyi_pfa.columns
])

Merged Junyi data: (16217311, 7)
Ability columns present: ['theta_all', 'theta_quartile', 'theta_sd_group']


## Sort data chronologically

In [ ]:
junyi_pfa = junyi_pfa.sort_values(
    ["student_id", "skill_id", "time"],
    kind="stable"
).reset_index(drop=True)

print("Rows:", len(junyi_pfa))
print("Students:", junyi_pfa["student_id"].nunique())
print("Skills:", junyi_pfa["skill_id"].nunique())
print("Time range:", junyi_pfa["time"].min(), "to", junyi_pfa["time"].max())

Rows: 16217311
Students: 72758
Skills: 1326
Time range: 2018-08-01 07:45:00+00:00 to 2019-08-01 00:00:00+00:00


## Create batches in order to not exhaust memory

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq

JUNYI_PFA_OUTPUT = os.path.join(
    BASE_DIR, "Junyi", "pfa_events_with_history.parquet"
)

# Remove a previous output so we don't accidentally append to stale results.
if os.path.exists(JUNYI_PFA_OUTPUT):
    os.remove(JUNYI_PFA_OUTPUT)

junyi_student_ids = junyi_pfa["student_id"].drop_duplicates().to_numpy()
JUNYI_BATCH_SIZE = 500
junyi_n_batches = (
    len(junyi_student_ids) + JUNYI_BATCH_SIZE - 1
) // JUNYI_BATCH_SIZE

print("Students:", len(junyi_student_ids))
print("Batch size:", JUNYI_BATCH_SIZE)
print("Total batches:", junyi_n_batches)
print("Output:", JUNYI_PFA_OUTPUT)

Students: 72758
Batch size: 500
Total batches: 146
Output: /content/drive/MyDrive/education-ml-research/Junyi/pfa_events_with_history.parquet


## Calculate history

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

output_path = JUNYI_PFA_OUTPUT
if os.path.exists(output_path):
    os.remove(output_path)

writer = None

try:
    for batch_num, start in enumerate(
        range(0, len(junyi_student_ids), JUNYI_BATCH_SIZE), start=1
    ):
        batch_ids = set(
            junyi_student_ids[start:start + JUNYI_BATCH_SIZE]
        )

        batch = junyi_pfa.loc[
            junyi_pfa["student_id"].isin(batch_ids),
            ["student_id", "skill_id", "correct", "time"]
        ].copy()

        keys = ["student_id", "skill_id", "time"]

        # Aggregate tied timestamps so none of those interactions
        # count as prior attempts for another tied interaction.
        history = (
            batch.groupby(keys, sort=False, observed=True)
            .agg(
                attempts_at_time=("correct", "size"),
                successes_at_time=("correct", "sum")
            )
            .reset_index()
        )
        history["failures_at_time"] = (
            history["attempts_at_time"] - history["successes_at_time"]
        )

        history = history.sort_values(keys, kind="stable")
        grouped = history.groupby(
            ["student_id", "skill_id"], sort=False, observed=True
        )

        history["prior_successes"] = (
            grouped["successes_at_time"].cumsum()
            - history["successes_at_time"]
        )
        history["prior_failures"] = (
            grouped["failures_at_time"].cumsum()
            - history["failures_at_time"]
        )

        batch = batch.merge(
            history[keys + ["prior_successes", "prior_failures"]],
            on=keys,
            how="left",
            validate="many_to_one",
            sort=False
        )

        table = pa.Table.from_pandas(batch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)

        del batch, history, grouped, table, batch_ids
        gc.collect()

        if batch_num % 10 == 0 or batch_num == junyi_n_batches:
            print(f"Completed batch {batch_num}/{junyi_n_batches}")

finally:
    if writer is not None:
        writer.close()

print("Saved:", output_path)

Completed batch 10/146
Completed batch 20/146
Completed batch 30/146
Completed batch 40/146
Completed batch 50/146
Completed batch 60/146
Completed batch 70/146
Completed batch 80/146
Completed batch 90/146
Completed batch 100/146
Completed batch 110/146
Completed batch 120/146
Completed batch 130/146
Completed batch 140/146
Completed batch 146/146
Saved: /content/drive/MyDrive/education-ml-research/Junyi/pfa_events_with_history.parquet


## Check batches

In [ ]:
import pyarrow.parquet as pq

junyi_pfa_file = pq.ParquetFile(JUNYI_PFA_OUTPUT)

print("Saved rows:", junyi_pfa_file.metadata.num_rows)
print("Saved row groups:", junyi_pfa_file.num_row_groups)
print("Saved columns:", junyi_pfa_file.schema.names)
print("Original rows:", len(junyi_pfa))

import pyarrow.parquet as pq
import numpy as np

parquet_file = pq.ParquetFile(JUNYI_PFA_OUTPUT)

total_rows = 0
missing_history = 0
negative_history = 0
max_successes = 0
max_failures = 0
sum_successes = 0
sum_failures = 0

for record_batch in parquet_file.iter_batches(
    batch_size=250_000,
    columns=["prior_successes", "prior_failures"]
):
    successes = record_batch.column("prior_successes").to_numpy()
    failures = record_batch.column("prior_failures").to_numpy()

    total_rows += len(successes)
    missing_history += int(
        np.isnan(successes).sum() + np.isnan(failures).sum()
    )
    negative_history += int(
        (successes < 0).sum() + (failures < 0).sum()
    )
    max_successes = max(max_successes, int(successes.max()))
    max_failures = max(max_failures, int(failures.max()))
    sum_successes += successes.sum()
    sum_failures += failures.sum()

print("Rows checked:", total_rows)
print("Missing history values:", missing_history)
print("Negative history values:", negative_history)
print("Maximum prior successes:", max_successes)
print("Maximum prior failures:", max_failures)
print("Mean prior successes:", round(sum_successes / total_rows, 3))
print("Mean prior failures:", round(sum_failures / total_rows, 3))

Saved rows: 16217311
Saved row groups: 146
Saved columns: ['student_id', 'skill_id', 'correct', 'time', 'prior_successes', 'prior_failures']
Original rows: 16217311
Rows checked: 16217311
Missing history values: 0
Negative history values: 0
Maximum prior successes: 1679
Maximum prior failures: 416
Mean prior successes: 3.391
Mean prior failures: 2.477


In [ ]:
from scipy import sparse
from sklearn.model_selection import train_test_split

# Confirm the student-level split
print("Train students:", len(junyi_train_set))
print("Test students:", len(junyi_test_set))
print("Student overlap:", len(junyi_train_set & junyi_test_set))

# Create a consistent index for each skill
junyi_skills = sorted(junyi_pfa["skill_id"].unique())
junyi_skill_to_idx = {
    skill: idx for idx, skill in enumerate(junyi_skills)
}
junyi_n_skills = len(junyi_skills)

print("Unique skills:", junyi_n_skills)
print("PFA features:", 3 * junyi_n_skills)

Train students: 58206
Test students: 14552
Student overlap: 0
Unique skills: 1326
PFA features: 3978


## Build matrix

In [ ]:
import pyarrow.parquet as pq
from scipy import sparse
import numpy as np

junyi_parquet = pq.ParquetFile(JUNYI_PFA_OUTPUT)

junyi_train_rows = 0
junyi_test_rows = 0

for rb in junyi_parquet.iter_batches(
    batch_size=250_000,
    columns=["student_id"]
):
    student_ids = rb.column("student_id").to_pylist()
    train_mask = np.fromiter(
        (sid in junyi_train_set for sid in student_ids),
        dtype=bool,
        count=len(student_ids)
    )
    junyi_train_rows += int(train_mask.sum())
    junyi_test_rows += int((~train_mask).sum())

print("Training rows:", junyi_train_rows)
print("Test rows:", junyi_test_rows)
print("Total rows:", junyi_train_rows + junyi_test_rows)

Training rows: 12914657
Test rows: 3302654
Total rows: 16217311


In [ ]:
def make_junyi_pfa_matrix(frame, skill_to_idx, n_skills):
    skill_idx = frame["skill_id"].map(skill_to_idx)

    if skill_idx.isna().any():
        raise ValueError("Found a skill without a mapped index.")

    skill_idx = skill_idx.to_numpy(dtype=np.int32)
    row_idx = np.arange(len(frame), dtype=np.int32)

    success = frame["prior_successes"].to_numpy(dtype=np.float64)
    failure = frame["prior_failures"].to_numpy(dtype=np.float64)

    rows = np.concatenate([row_idx, row_idx, row_idx])
    cols = np.concatenate([
        skill_idx,
        n_skills + skill_idx,
        2 * n_skills + skill_idx
    ])
    values = np.concatenate([
        np.ones(len(frame), dtype=np.float64),
        success,
        failure
    ])

    return sparse.coo_matrix(
        (values, (rows, cols)),
        shape=(len(frame), 3 * n_skills)
    ).tocsr()

print("Junyi PFA sparse-matrix function defined.")

Junyi PFA sparse-matrix function defined.


## Stream the batches to save memory

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from scipy.sparse import save_npz, load_npz

JUNYI_MATRIX_DIR = os.path.join(BASE_DIR, "Junyi", "pfa_matrices")
os.makedirs(JUNYI_MATRIX_DIR, exist_ok=True)

# One row per student: compact lookup for ability groups
junyi_group_lookup = (
    ability_junyi[["uuid", "theta_quartile", "theta_sd_group"]]
    .rename(columns={"uuid": "student_id"})
    .copy()
)
junyi_group_lookup["student_id"] = junyi_group_lookup["student_id"].astype(str)
junyi_group_lookup = junyi_group_lookup.set_index("student_id")

train_parts_dir = os.path.join(JUNYI_MATRIX_DIR, "train_parts")
test_parts_dir = os.path.join(JUNYI_MATRIX_DIR, "test_parts")
os.makedirs(train_parts_dir, exist_ok=True)
os.makedirs(test_parts_dir, exist_ok=True)

# Clear previous partial outputs
for folder in [train_parts_dir, test_parts_dir]:
    for name in os.listdir(folder):
        if name.endswith(".npz"):
            os.remove(os.path.join(folder, name))

train_labels = []
test_labels = []
train_quartiles = []
test_quartiles = []
train_sd_groups = []
test_sd_groups = []

parquet_file = pq.ParquetFile(JUNYI_PFA_OUTPUT)
train_rows = 0
test_rows = 0

for batch_num, rb in enumerate(
    parquet_file.iter_batches(
        batch_size=100_000,
        columns=[
            "student_id", "skill_id", "correct",
            "prior_successes", "prior_failures"
        ]
    ),
    start=1
):
    frame = rb.to_pandas()
    frame["student_id"] = frame["student_id"].astype(str)

    groups = junyi_group_lookup.reindex(frame["student_id"])
    frame["theta_quartile"] = groups["theta_quartile"].to_numpy()
    frame["theta_sd_group"] = groups["theta_sd_group"].to_numpy()

    if frame[["theta_quartile", "theta_sd_group"]].isna().any().any():
        raise ValueError(f"Missing ability groups in batch {batch_num}")

    train_mask = frame["student_id"].isin(junyi_train_set).to_numpy()
    test_mask = ~train_mask

    for mask, split_name in [
        (train_mask, "train"),
        (test_mask, "test")
    ]:
        if not mask.any():
            continue

        part = frame.loc[mask]
        X_part = make_junyi_pfa_matrix(
            part, junyi_skill_to_idx, junyi_n_skills
        )
        y_part = part["correct"].to_numpy(dtype=np.int8)

        part_path = os.path.join(
            train_parts_dir if split_name == "train" else test_parts_dir,
            f"batch_{batch_num:03d}.npz"
        )
        save_npz(part_path, X_part, compressed=False)

        if split_name == "train":
            train_labels.append(y_part)
            train_quartiles.extend(part["theta_quartile"].tolist())
            train_sd_groups.extend(part["theta_sd_group"].tolist())
            train_rows += len(part)
        else:
            test_labels.append(y_part)
            test_quartiles.extend(part["theta_quartile"].tolist())
            test_sd_groups.extend(part["theta_sd_group"].tolist())
            test_rows += len(part)

    del frame, groups
    gc.collect()

    if batch_num % 25 == 0:
        print(f"Processed {batch_num} batches")

# Save labels and group labels
np.save(os.path.join(JUNYI_MATRIX_DIR, "y_train.npy"),
        np.concatenate(train_labels))
np.save(os.path.join(JUNYI_MATRIX_DIR, "y_test.npy"),
        np.concatenate(test_labels))

pd.DataFrame({
    "theta_quartile": train_quartiles,
    "theta_sd_group": train_sd_groups
}).to_csv(os.path.join(JUNYI_MATRIX_DIR, "train_groups.csv"), index=False)

pd.DataFrame({
    "theta_quartile": test_quartiles,
    "theta_sd_group": test_sd_groups
}).to_csv(os.path.join(JUNYI_MATRIX_DIR, "test_groups.csv"), index=False)

print("Training rows:", train_rows)
print("Test rows:", test_rows)
print("Total rows:", train_rows + test_rows)
print("Saved batch matrices to:", JUNYI_MATRIX_DIR)

Processed 25 batches
Processed 50 batches
Processed 75 batches
Processed 100 batches
Processed 125 batches
Processed 150 batches
Training rows: 12914657
Test rows: 3302654
Total rows: 16217311
Saved batch matrices to: /content/drive/MyDrive/education-ml-research/Junyi/pfa_matrices


## Fit the model

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.sparse import load_npz
import numpy as np
import pandas as pd
import os
import gc

JUNYI_MATRIX_DIR = os.path.join(BASE_DIR, "Junyi", "pfa_matrices")
train_parts_dir = os.path.join(JUNYI_MATRIX_DIR, "train_parts")
test_parts_dir = os.path.join(JUNYI_MATRIX_DIR, "test_parts")

y_train_junyi = np.load(os.path.join(JUNYI_MATRIX_DIR, "y_train.npy"))
y_test_junyi = np.load(os.path.join(JUNYI_MATRIX_DIR, "y_test.npy"))

train_groups = pd.read_csv(
    os.path.join(JUNYI_MATRIX_DIR, "train_groups.csv")
)
test_groups = pd.read_csv(
    os.path.join(JUNYI_MATRIX_DIR, "test_groups.csv")
)

def predict_saved_batches(model, folder, y_true):
    files = sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".npz")
    )
    predictions = []
    offset = 0

    for file_path in files:
        X_batch = load_npz(file_path)
        n = X_batch.shape[0]
        predictions.append(model.predict_proba(X_batch)[:, 1])
        offset += n
        del X_batch
        gc.collect()

    probabilities = np.concatenate(predictions)

    if len(probabilities) != len(y_true):
        raise ValueError(
            f"Prediction count {len(probabilities)} != label count {len(y_true)}"
        )

    return probabilities

# Generate predictions
p_train_junyi = predict_saved_batches(
    pfa_junyi, train_parts_dir, y_train_junyi
)
p_test_junyi = predict_saved_batches(
    pfa_junyi, test_parts_dir, y_test_junyi
)

def summarize_metrics(y, p, groups, split_name, group_column=None):
    if group_column is None:
        return [{
            "Split": split_name,
            "Ability group": "Overall",
            "Responses": len(y),
            "Students": None,
            "AUC": roc_auc_score(y, p),
            "Brier Score": brier_score_loss(y, p),
            "Correct rate": np.mean(y)
        }]

    results = []
    for group in groups[group_column].dropna().unique():
        mask = groups[group_column].to_numpy() == group
        y_group = y[mask]
        p_group = p[mask]

        results.append({
            "Split": split_name,
            "Ability group": group,
            "Responses": int(mask.sum()),
            "Students": None,
            "AUC": (
                roc_auc_score(y_group, p_group)
                if len(np.unique(y_group)) == 2 else np.nan
            ),
            "Brier Score": brier_score_loss(y_group, p_group),
            "Correct rate": np.mean(y_group)
        })

    return results

results = []
for split_name, y, p, groups in [
    ("Train", y_train_junyi, p_train_junyi, train_groups),
    ("Test", y_test_junyi, p_test_junyi, test_groups)
]:
    results.extend(summarize_metrics(y, p, groups, split_name))

    for col in ["theta_quartile", "theta_sd_group"]:
        results.extend(
            summarize_metrics(y, p, groups, split_name, col)
        )

junyi_pfa_results = pd.DataFrame(results)
display(junyi_pfa_results.round(4))

,Split,Ability group,Responses,Students,AUC,Brier Score,Correct rate
0,Train,Overall,12914657,None,0.7014,0.1868,0.7051
1,Train,Q4,2355545,None,0.6861,0.1470,0.8362
2,Train,Q1,2777787,None,0.7072,0.2385,0.5551
3,Train,Q2,4094741,None,0.7141,0.1927,0.6823
4,Train,Q3,3686584,None,0.7072,0.1668,0.7598
5,Train,> +1.5 SD,235569,None,0.6754,0.1290,0.9115
6,Train,< -1.5 SD,467613,None,0.6866,0.2845,0.4323
7,Train,-1.0 to -0.5 SD,2526380,None,0.7144,0.2096,0.6340
8,Train,-0.5 to 0 SD,3439391,None,0.7127,0.1853,0.7038
9,Train,-1.5 to -1.0 SD,1157001,None,0.7085,0.2409,0.5476



## Junyi Academy — PFA Results

### Overall Results

| Split | Responses | AUC | Brier Score | Correct Rate |
|---|---:|---:|---:|---:|
| Train | 12,914,657 | 0.7014 | 0.1868 | 0.7051 |
| Test | 3,302,654 | 0.7026 | 0.1889 | 0.6982 |

### Test Results by IRT Ability Quartile

| Ability Group | Responses | Students | AUC | Brier Score | Correct Rate |
|---|---:|---:|---:|---:|---:|
| Q1 (lowest ability) | 723,479 | 95 | 0.7120 | 0.2381 | 0.5475 |
| Q2 | 1,034,722 | 73 | 0.7119 | 0.1968 | 0.6710 |
| Q3 | 933,596 | 94 | 0.7064 | 0.1684 | 0.7564 |
| Q4 (highest ability) | 610,857 | 80 | 0.6847 | 0.1485 | 0.8337 |

### Test Results by IRT Standard-Deviation Group

| Ability Group | Responses | Students | AUC | Brier Score | Correct Rate |
|---|---:|---:|---:|---:|---:|
| Below −1.5 SD | 119,537 | 32 | 0.6988 | 0.2806 | 0.4236 |
| −1.5 to −1.0 SD | 286,744 | 66 | 0.7188 | 0.2377 | 0.5481 |
| −1.0 to −0.5 SD | 662,671 | 74 | 0.7129 | 0.2131 | 0.6238 |
| −0.5 to 0 SD | 879,607 | 71 | 0.7145 | 0.1898 | 0.6894 |
| 0 to +0.5 SD | 714,407 | 42 | 0.7035 | 0.1662 | 0.7644 |
| +0.5 to +1.0 SD | 417,931 | 27 | 0.6930 | 0.1527 | 0.8132 |
| +1.0 to +1.5 SD | 164,485 | 16 | 0.6738 | 0.1481 | 0.8490 |
| Above +1.5 SD | 57,272 | 14 | 0.6794 | 0.1227 | 0.9207 |

### Interpretation

The pooled PFA model achieved a test AUC of **0.7026** and a Brier Score of **0.1889**. Across the IRT quartiles, test AUC ranged from 0.6847 to 0.7120, while Brier Scores generally decreased as ability increased. This pattern is consistent with differences in probabilistic prediction error across ability groups, although the Brier Score is also affected by differences in each group's correct-response rate.

The standard-deviation analysis showed a similar broad pattern in Brier Scores, but AUC did not increase consistently with ability. Results for the most extreme groups should be interpreted cautiously because some groups contain relatively few test students.

These findings describe the performance of this PFA implementation on Junyi Academy. They do not, by themselves, establish that ability caused the observed performance differences or that the same pattern will hold for other datasets or knowledge-tracing models.
